In [271]:
%reset -f

In [272]:
typ = "example"  # Type of simulation, used for naming files

In [273]:

import xml.etree.ElementTree as ET
import mujoco
import csv
import open3d as o3d
from sim_fxn_lib import *
import numpy as np
import os
import sys
import pickle
import scipy


In [274]:
v = {}

In [275]:
# load in the mujoco model
xml_path = 'RHex1-water-rigid.xml'
tMax = 3
dt = 0.001
model, data, renderer, t, dt, frames, framerate, sand_h_id, stl_path = initialize_simulation(xml_path=xml_path, stl='sandflipper.stl', tMax=tMax, dt=dt)
numSteps = len(t)
tMax = t[-1]
renderer.update_scene(data, camera="diag") # can replace with other cameras
scene_option = mujoco.MjvOption() # these two scene lines make the contact forces visible in the sim
scene_option.flags[mujoco.mjtVisFlag.mjVIS_CONTACTFORCE] = True


In [276]:
# this grabs the mujoco IDs for the position controller actuators

pos_actuator_ids = {}
actuator_names = [
    "front right_p", "front left_p",
    "middle right_p", "middle left_p",
    "back right_p", "back left_p"
]

# Loop through each name and get the corresponding ID
for name in actuator_names:
    pos_actuator_ids[name] = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_ACTUATOR, name)

fr_pos_id = pos_actuator_ids["front right_p"]
mr_pos_id = pos_actuator_ids["middle right_p"]
br_pos_id = pos_actuator_ids["back right_p"]
fl_pos_id = pos_actuator_ids["front left_p"]
ml_pos_id = pos_actuator_ids["middle left_p"]
bl_pos_id = pos_actuator_ids["back left_p"]
# distal_fr_vert_pos_id = pos_actuator_ids["distal segment front right_vert"]

print("Actuator IDs:")
for name, actuator_id in pos_actuator_ids.items():
    print(f"{name}: {actuator_id}")

Actuator IDs:
front right_p: 0
front left_p: 3
middle right_p: 1
middle left_p: 4
back right_p: 2
back left_p: 5


In [277]:
# Initialize joint ID dictionary
joint_ids = {}

# Define the desired initial angles in degrees
initial_angles = {
    "mid segment front left": 30, "distal segment front left": 45,
    "mid segment middle left": 30, "distal segment middle left": 45,
    "mid segment back left": 30, "distal segment back left": 45,
    "mid segment front right": 30, "distal segment front right": 45,
    "mid segment middle right": 30, "distal segment middle right": 45,
    "mid segment back right": 30, "distal segment back right": 45,
}

# Populate joint_ids with IDs for each joint name
for joint_name, angle_deg in initial_angles.items():
    try:
        joint_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_JOINT, joint_name)
        joint_ids[joint_name] = joint_id
    except mujoco.Error:
        print(f"Warning: Joint '{joint_name}' not found in model.")

mujoco.mj_forward(model, data)


In [278]:
# site_name_sand = "sand_height"
# sand_h_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_SITE, site_name_sand)

# # Front right leg
# tip_fr_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_SITE, "tip_fr")
# com_id_dfr = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_SITE, "com_distal front right")
# body_id_dsfr = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_BODY, "distal segment front right")

# # Front left leg
# tip_fl_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_SITE, "tip_fl")
# com_id_dfl = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_SITE, "com_distal front left")
# body_id_dsfl = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_BODY, "distal segment front left")

# # Middle right leg
# tip_mr_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_SITE, "tip_mr")
# com_id_dmr = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_SITE, "com_distal middle right")
# body_id_dsmr = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_BODY, "distal segment middle right")

# # Middle left leg
# tip_ml_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_SITE, "tip_ml")
# com_id_dml = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_SITE, "com_distal middle left")
# body_id_dsml = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_BODY, "distal segment middle left")

# # Rear right leg
# tip_rr_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_SITE, "tip_rr")
# com_id_drr = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_SITE, "com_distal rear right")
# body_id_dsrr = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_BODY, "distal segment rear right")


# # back left leg
# tip_bl_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_SITE, "tip_bl")
# com_id_dbl = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_SITE, "com_distal back left")
# body_id_dsbl = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_BODY, "distal segment back left")


In [279]:


body, vertices, faces, mesh = load_and_process_mesh(stl_path, scale_factor=1000)

In [280]:

entities = get_named_bodies_from_xml(xml_path)


In [281]:
for entity in entities:
    v[f'fm_{entity}'] = []
    v[f'mm_{entity}'] = []

motion_data = {
    "time": [],
    "x": [],
    "y": [],
    "z": [],
    "roll": [],
    "pitch": [],
    "yaw": []
}

In [282]:



camera_list = ["diag"]
frames = {cam: [] for cam in camera_list}
combined_framesFR = []
frames_diag = []


In [283]:
# F_full_sorted_prev = np.zeros_like(faces)  # This will be updated properly later

# # For each body, store previous state and geometry
# prev_body_pos_dict = {name: None for name in entities}
# prev_body_quat_dict = {name: None for name in entities}

# vertices_dict = {}
# faces_dict = {}
# body_dict = {}
# mesh_dict = {}
# for body_name in entities:
#     stl_path = f"C:\\Users\\Crab\\Documents\\Repos\\sand_sim\\asset\\{body_name}.stl"

#     # Load mesh from STL
#     body, vertices, faces, mesh = load_and_process_mesh(stl_path, scale_factor=1000)
#     vertices_dict[body_name] = vertices
#     faces_dict[body_name] = faces
#     body_dict[body_name] = body
#     mesh_dict[body_name] = mesh

# # Results
# v = {}
# for body_name in entities:
#     v[f'fm_{body_name}'] = []
#     v[f'mm_{body_name}'] = []

# def initialize_sites_for_all_bodies(model, data, body_dict, sitename="force"):
#     """Initialize site positions for multiple bodies"""
#     for body_name, mesh in mesh_dict.items():
#         initialize_sites_on_mesh(
#             model=model,
#             data=data,
#             mesh=mesh,
#             sitename=f"{sitename}_{body_name}",
#             bodyname=body_name
#         )
        
# initialize_sites_for_all_bodies(model, data, body_dict, sitename="force")

In [284]:
# def generate_site_lines(body_names, sites_per_body=500, output_path="sites.txt"):
#     with open(output_path, "w") as f:
#         for body_name in body_names:
#             for i in range(sites_per_body):
#                 line = f'<site name="force_{body_name}_site_{i}" pos="0 0 0.000001" size="0.0009" type="sphere" rgba="1 1 1 1"/>\n'
#                 f.write(line)
#     print(f"Site definitions written to {output_path}")

# if __name__ == "__main__":
#     body_list = entities  # example
#     generate_site_lines(body_list, sites_per_body=500, output_path="sites.txt")


In [285]:
i = 0

In [286]:
import mujoco
import mujoco.viewer
import time
import numpy as np
import scipy.spatial.transform
applied_force = True
mass = {}
save_every = 6
for body_name in entities:
    body_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_BODY, body_name)

    mass[body_name] = model.body_mass[body_id]

import os, cv2                      # ➊ use OS & OpenCV -Sophia
os.makedirs("frames", exist_ok=True) # ➋ ensure output folder exists -Sophia

# Reset simulation
model.opt.timestep = dt  # match external dt
once_submerged = False  # Flag to check if any body has submerged
# with mujoco.viewer.launch_passive(model, data) as viewer:
print(len(t))


# dactyl_sinkage = {name: [] for name in [
#     "prox_fr_dactyl_tip",
#     "prox_mr_dactyl_tip",
#     "prox_br_dactyl_tip",
#     "prox_fl_dactyl_tip",
#     "prox_ml_dactyl_tip",
#     "prox_bl_dactyl_tip"
# ]}
# dactyl_site_ids = {name: mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_SITE, name)
#                    for name in dactyl_sinkage.keys()}

plate_pos = []


3000


In [287]:

for i in range(len(t)):

    if i < 1000:
        data.ctrl[pos_actuator_ids["front right_p"]] = np.pi
        data.ctrl[pos_actuator_ids["middle right_p"]] = 0
        data.ctrl[pos_actuator_ids["back right_p"]] = np.pi
        data.ctrl[pos_actuator_ids["front left_p"]] = 0
        data.ctrl[pos_actuator_ids["middle left_p"]] = np.pi
        data.ctrl[pos_actuator_ids["back left_p"]] = 0
    else:
        data.ctrl[pos_actuator_ids["front right_p"]] = -2*np.pi*(t[i-1000]) + 0.75*np.sin(2*np.pi*t[i-1000]) + np.pi
        data.ctrl[pos_actuator_ids["middle right_p"]] = -2*np.pi*(t[i-1000]) + 0.75*np.sin(2*np.pi*t[i-1000] + np.pi)
        data.ctrl[pos_actuator_ids["back right_p"]] = -2*np.pi*(t[i-1000]) + 0.75*np.sin(2*np.pi*t[i-1000]) + np.pi
        data.ctrl[pos_actuator_ids["front left_p"]] = -2*np.pi*(t[i-1000]) + 0.75*np.sin(2*np.pi*t[i-1000] + np.pi)
        data.ctrl[pos_actuator_ids["middle left_p"]] = -2*np.pi*(t[i-1000]) + 0.75*np.sin(2*np.pi*t[i-1000]) + np.pi
        data.ctrl[pos_actuator_ids["back left_p"]] = -2*np.pi*(t[i-1000]) + 0.75*np.sin(2*np.pi*t[i-1000] + np.pi)

    mujoco.mj_step(model, data)
    # print(f"applied force before clearing: {data.qfrc_applied}") # DEBUG
    # data.qfrc_applied[:] = 0  # fix in future
    # Loop through bodies
    # body_list = ["distal segment front left"]
    # global_pos_sand = data.site_xpos[sand_h_id]
    
    # for body_name in body_list:
        # if body_name == "plate":
        #     plate_pos.append(np.array(data.xpos[body_id]))
   
        # v[f'fm_{body_name}'].append(np.array([0, 0, 0]))
        # v[f'mm_{body_name}'].append(np.array([0, 0, 0]))
    if i % save_every == 0:
       
        renderer.update_scene(data, camera="diag")
        frame = renderer.render()
        cv2.imwrite(f"frames/{typ}_rigid_frame_{i:04d}.png", frame)
        cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
  

In [288]:
with open(f"W{typ}_plate_position.csv", "w", newline="") as f:

    writer = csv.writer(f)
    writer.writerow(["X [m]", "Y [m]", "Z [m]"])  # header
    for i in range(len(plate_pos)):
        row = [f"{coord:.4f}" for coord in plate_pos[i]]
        writer.writerow(row)

In [289]:
def create_video_from_frames(typ, frame_folder, frame_prefix, save_every, output_name=None):
    fps = 1000 / save_every  # Match the simulation timestep
    if output_name is None:
        output_video = f"{typ}_rigid.mp4"
    else:
        output_video = output_name
    frames = sorted([f for f in os.listdir(frame_folder) 
                     if f.startswith(frame_prefix) and f.endswith(".png")])
    first_frame = cv2.imread(os.path.join(frame_folder, frames[0]))
    height, width, _ = first_frame.shape
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_video, fourcc, fps, (width, height))
    # Write all frames to video
    for f in frames:
        img = cv2.imread(os.path.join(frame_folder, f))
        if img.ndim == 2:
            img = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
        out.write(img)
    out.release()
    print(f"Video saved to {output_video}")
    return output_video

In [290]:
create_video_from_frames(
    typ=typ,
    frame_folder=r"frames",
    frame_prefix=f"{typ}_rigid_frame_",
    save_every=save_every,
)

Video saved to example_rigid.mp4


'example_rigid.mp4'